<a href="https://www.kaggle.com/code/shreyashr2406/task-2-genai-langchain-deep-technical-blog?scriptVersionId=311160259" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

1) Installation

In [1]:
!pip install -qU langchain langchain-groq langchain-community langchain-core langgraph langchain-huggingface faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google

2) Set up Groq API key 

In [2]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

3. Basic LLM Call

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

response = llm.invoke("Explain LangChain in one line")
print(response.content)

LangChain is an open-source framework that enables developers to build applications powered by large language models, providing a set of tools and APIs to interact with and fine-tune these models.


4. PromptTemplate usage

In [4]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Explain {topic} in simple terms")
formatted_prompt = prompt.format(topic="LangChain")

print(f"Prompt: {formatted_prompt}\n")
print("--- LLM Output ---")
print(llm.invoke(formatted_prompt).content)

Prompt: Explain LangChain in simple terms

--- LLM Output ---
LangChain is a framework that helps developers build applications using large language models (LLMs) like chatbots, virtual assistants, or other AI-powered tools. 

Imagine you have a super smart, magic notebook that can understand and respond to anything you write in it. This notebook is like a large language model. However, to make the notebook really useful, you need to teach it how to do specific tasks, like answering questions, generating text, or even controlling other devices.

LangChain is like a set of instructions that helps you teach the magic notebook (or the large language model) how to do these tasks. It provides a way to break down complex tasks into smaller, manageable pieces, and then use the language model to complete each piece.

The "chain" part of LangChain refers to the idea of linking together multiple language models, or using a single model in a series of steps, to achieve a specific goal. This allow

5. Simple Chain

In [5]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "AI Agents"})
print(result)

**What is an AI Agent.**
An AI agent is a computer program that can perform tasks on its own, using artificial intelligence (AI) to make decisions and take actions. Think of it like a robot that can think and act for itself.

**Key Characteristics:**

1. **Autonomy**: AI agents can work independently, without human intervention.
2. **Decision-making**: They can make decisions based on the information they have and the goals they're trying to achieve.
3. **Action**: They can take actions to achieve their goals, like moving, communicating, or solving problems.
4. **Learning**: Many AI agents can learn from their experiences and improve over time.

**Types of AI Agents:**

1. **Simple Agents**: These agents follow pre-programmed rules to perform tasks, like a thermostat that turns on the heat when it's cold.
2. **Intelligent Agents**: These agents can learn and adapt to new situations, like a self-driving car that adjusts its route based on traffic.
3. **Multi-Agent Systems**: These are g

6. Agent with tool

In [6]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def convert_usd_to_inr(usd: float) -> str:
    """Convert USD to INR using a fixed exchange rate of 1 USD = 83 INR."""
    inr = usd * 83
    return f"${usd:.2f} is approximately ₹{inr:.2f}"

@tool
def calculate_emi(principal: float, annual_rate: float, months: int) -> str:
    """Calculate monthly EMI for a loan using principal, annual interest rate, and duration."""
    monthly_rate = annual_rate / 12 / 100
    if monthly_rate == 0:
        emi = principal / months
    else:
        emi = principal * monthly_rate * (1 + monthly_rate)**months / ((1 + monthly_rate)**months - 1)
    return f"Monthly EMI: ₹{emi:.2f}"

tools = [convert_usd_to_inr, calculate_emi]

# Create the agent
agent = create_react_agent(model=llm, tools=tools)

# Run the agent
result = agent.invoke({
    "messages": [("human", "Convert 250 USD to INR and calculate the EMI for a ₹500000 loan at 10% for 24 months.")]
})

print("=== Final Answer ===")
print(result["messages"][-1].content)

/tmp/ipykernel_23/2792736027.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools)


=== Final Answer ===
The conversion of 250 USD to INR is approximately ₹20750.00. The monthly EMI for a ₹500000 loan at 10% for 24 months is ₹20855.04.


7. Memory example

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt_with_history | llm | StrOutputParser()
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_1"}}
resp1 = with_message_history.invoke({"input": "Hi, my name is Sai Kadiravan."}, config=config)
print(f"Assistant: {resp1}")

resp2 = with_message_history.invoke({"input": "What is my name?"}, config=config)
print(f"Assistant: {resp2}")

Assistant: Hello Sai Kadiravan, it's nice to meet you. Is there something I can help you with or would you like to chat?
Assistant: Your name is Sai Kadiravan.


Vector Store Implementation (FAISS + HuggingFace)

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough

# 1. Knowledge Base
tech_docs = [
    Document(page_content="The Model Context Protocol (MCP) allows AI agents to interact with local tools safely."),
    Document(page_content="LangChain's LCEL allows for declarative composition of chains.")
]

# 2. Split and Embed
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
chunks = splitter.split_documents(tech_docs)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# 3. RAG Chain
rag_template = """Answer the question based only on the context:
{context}

Question: {question}"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

rag_chain = (
    {"context": vector_db.as_retriever(), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("--- RAG Result ---")
print(rag_chain.invoke("How does MCP help AI agents?"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- RAG Result ---
MCP (Model Context Protocol) helps AI agents by allowing them to interact with local tools safely.


Real-World Use Case: Personal Technical Library Assistant

In [9]:
# Real-World Use Case: RAG-based Technical Assistant
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# 1. Setup the Model (Llama-3.1-8b) & Embeddings
# Note: Using 8b-instant for fast RAG responses
llm_rag = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. The Knowledge Base (Your personal technical library)
tech_docs = [
    Document(page_content="The Model Context Protocol (MCP) allows AI agents to interact with local tools safely."),
    Document(page_content="In NLP, Token Classification is used for Named Entity Recognition (NER)."),
    Document(page_content="LangChain's LCEL (LangChain Expression Language) allows for declarative composition of chains."),
    Document(page_content="Agentic AI refers to systems where LLMs use tools and reasoning to achieve a goal.")
]

# 3. Component: Indexing (Splitting + Vector Store)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
doc_chunks = text_splitter.split_documents(tech_docs)
vector_db = FAISS.from_documents(doc_chunks, embeddings)

# 4. Component: The RAG Chain (Orchestration)
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Modern LCEL Chain construction
rag_chain = (
    {"context": vector_db.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm_rag
    | StrOutputParser()
)

# 5. Execution
print("--- Personal Technical Assistant ---")
try:
    query = "How does MCP help AI agents?"
    response = rag_chain.invoke(query)
    print(f"Query: {query}")
    print(f"Response: {response}")
except Exception as e:
    print(f"Error: {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Personal Technical Assistant ---
Query: How does MCP help AI agents?
Response: The Model Context Protocol (MCP) allows AI agents to interact with local tools safely.
